# V2 — Variance replication, OHLC estimators and VIX

**Audience:** analysts comfortable with Black-Scholes, volatility surfaces and realized returns.

**Outcome:** reconstruct a variance swap from an option strip, distinguish variance and volatility units, compare OHLC estimators, and price VIX futures/options. The dense fixed-volatility surface is synthetic, with enough wing coverage to make truncation negligible.

## Financial context and interpretation

### Variance units and payoff

Represent 20% volatility as `0.20` and its variance as `0.04`. A receive-variance swap with variance notional `N` pays `N*(realized_variance-strike_variance)`. The notional is dollars per one unit of **decimal variance**, not dollars per volatility point. Near strike volatility `K`, the vega-notional convention is `N_vega=N*2*K*0.01`, where one volatility point means one percentage point.

The exact quadratic payoff differs from a linear vega approximation when volatility moves materially. Its convexity residual is not a pricing error. The terminal exercise below compares realized 18% with strike 20%, making the distinction visible without mixing discounting into terminal P&L.

### Replication uses the whole option strip

The log-contract identity weights out-of-the-money puts below the forward and calls above it approximately by `1/K^2`. A discrete grid requires strike widths, an anchor strike just below the forward, and a correction for the forward/anchor gap. The native implementation uses the displayed exact logarithmic anchor correction and half-width endpoints. The code reproduces those conventions, so its agreement tests the actual library computation.

This is the supported `discounting` registry path for the variance instrument; there is no separate `static_replication` model key. The wide flat-volatility strip is deliberately synthetic. Exact agreement with the native strip does not eliminate finite-grid or wing-truncation error. We therefore compare the strike separately with the known flat-volatility-squared limit and show the remaining discretization difference.

Removing the put wing loses a positive replication contribution and biases the strike downward in this controlled example. A dense ATM smile alone cannot establish adequate wing coverage. Real markets also require quote screening, consistent expiries, forward estimation and bid/ask treatment. The financial exposure is a contract with stated observation and settlement conventions, not simply the output of an option integral.

### Realized estimators answer different questions

Close-to-close variance uses observed log returns. Parkinson uses the intraday high/low range. Garman-Klass and Rogers-Satchell also use open/close information, with different drift assumptions. Yang-Zhang incorporates overnight and intraday components. A gappy market can show substantial close-to-close or Yang-Zhang variation while an intraday-range estimator misses much of the overnight move.

Use aligned positive OHLC observations with highs and lows consistent with opens and closes. State whether the series is adjusted and what corporate-action treatment applies. The contract's chosen estimator is part of its payoff definition; analysts cannot select whichever estimator gives a preferred P&L after the fact.

The raw `variance_realized` statistic uses its observation-frequency annualization. A seasoned mark weights elapsed time on the contract's day-count basis. To combine them correctly, accrued squared log returns contribute `sum(log_returns^2)/total_years`, while the remaining forward variance contributes `remaining_years/total_years*forward_variance`. Multiplying an observation-annualized statistic by a calendar-time weight can break that identity. The 60%-elapsed proof reconciles the expected-variance metric to PV and checks the risk scaling directly.

### Vega, volatility swaps and VIX are distinct exposures

`variance_vega` is the change in PV per unit of forward decimal variance, scaled by the unobserved fraction. The native volatility vega near the assumed forward volatility equals `variance_vega*2*sigma*0.01`. As observations accrue, the forward-sensitive fraction declines. The strike-volatility metric is named `variance_strike_vol`; `variance_expected` and `variance_realized` are the other actual metric keys.

A volatility swap depends on expected realized volatility, which is generally less than the square root of expected realized variance. The two-state example calculates the Jensen gap and a second-order approximation explicitly. It is a transparent payoff calculation on disclosed probabilities, not a new native volatility-swap pricer or a calibration of volatility-of-volatility.

Cboe's VIX measures near-term volatility inferred from SPX options, while VIX futures refer to a future settlement of that index. A VIX future therefore need not equal today's spot VIX, and rolling a futures exposure does not reproduce an investment in the spot index. [Cboe VIX FAQ](https://www.cboe.com/tradable-products/vix/faqs).

The fixture's volatility-index curve stores index points, not decimal volatility. The native VIX listed-option envelope uses a forward index level and Black-76 terms. Its 60% option volatility describes uncertainty in that level; it is not a forecast that SPX realized volatility equals 60%. The fixed-curve roll calculation is a scenario showing movement along an unchanged curve, not a forecast of future settlement.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from datetime import date
import json
import math
import numpy as np
import pandas as pd
from _shared import analyst_tracks as tracks
from finstack_quant.valuations.instruments import price_instrument
AS_OF = tracks.AS_OF

## 1. Native expected variance and payoff units

The strike is 0.04, which equals 20% squared. A variance notional of $1m pays $1m times the difference in decimal variances. The native `discounting` model obtains implied variance from the surface named `SPX_VOL`, which is distinct from the ordinary SPX option surface in the common book.

In [ ]:
from finstack_quant.models import bs_price
market = tracks.build_market("vol")
inputs = tracks.variance_inputs()
result = price_instrument(json.dumps(inputs["SPX-VARIANCE"]), market, AS_OF,
    model="discounting", metrics=["variance_expected", "variance_realized"])
T, spot, dividend_yield = 1.0, 5200.0, 0.015
df = market.get_discount("USD-OIS").df(T)
r = -math.log(df)/T
native_variance = result.metrics["variance_expected"]
assert abs(result.value.amount - 1_000_000*df*(native_variance-0.04)) < 0.01
print(pd.Series({"expected_variance_decimal": native_variance, "implied_vol_percent": 100*math.sqrt(native_variance),
           "pv_usd": result.value.amount}))

## 2. Replicate the log contract with visible strip weights

Use puts below $K_0$, calls above it, and the average put/call price at $K_0$, the highest grid strike below the forward. Endpoint widths are half cells. The anchor correction is the exact log expression $2[(F/K_0-1)-\log(F/K_0)]/T$. The native engine also integrates flat-volatility wings; here ±3 log-moneyness makes their contribution negligible.

In [ ]:
strikes = np.array([5200.0*math.exp(-3.0+i*0.05) for i in range(121)])
forward = spot/df * math.exp(-dividend_yield*T)
k0_index = np.flatnonzero(strikes <= forward)[-1]
k0 = strikes[k0_index]
widths = np.empty(len(strikes))
widths[0], widths[-1] = (strikes[1]-strikes[0])/2, (strikes[-1]-strikes[-2])/2
widths[1:-1] = (strikes[2:]-strikes[:-2])/2
option_prices = []
for i, strike in enumerate(strikes):
    call = bs_price(spot, float(strike), r, dividend_yield, 0.20, T, True)
    put = bs_price(spot, float(strike), r, dividend_yield, 0.20, T, False)
    option_prices.append(put if i < k0_index else call if i > k0_index else (put+call)/2)
strip_variance = 2/(T*df)*np.sum(widths*np.array(option_prices)/strikes**2)
strip_variance -= 2/T*((forward/k0-1)-math.log(forward/k0))
assert abs(strip_variance-native_variance) < 1e-10
assert abs(native_variance-0.20**2) < 0.0005
print(pd.Series({"strip_variance": strip_variance, "native_variance": native_variance,
           "grid_discretization_vs_flat_vol_squared": native_variance-0.20**2}))

A discrete strip need not equal exactly $\sigma^2$ even on a flat surface: quadrature and the $K_0$ anchor matter. Matching the native strip and separately bounding the flat-volatility residual demonstrates both implementation agreement and numerical accuracy.

## 3. Compare realized OHLC estimates

All four OHLC series share dates. High/low ranges capture intraday activity that close-to-close returns omit; Yang-Zhang also uses overnight and intraday components. Different estimates are expected and do not imply that one series was silently rescaled.

In [ ]:
from finstack_quant.core.math import stats
observations = tracks.ohlc_observations()
arrays = {key: np.array([value for _, value in rows]) for key, rows in observations.items()}
estimates = {"close_to_close": stats.realized_variance(arrays["SPX-CLOSE"])}
for method in ["parkinson", "garman_klass", "rogers_satchell", "yang_zhang"]:
    estimates[method] = stats.realized_variance_ohlc(arrays["SPX-OPEN"], arrays["SPX-HIGH"],
        arrays["SPX-LOW"], arrays["SPX-CLOSE"], method=method, annualization_factor=252.0)
assert all(math.isfinite(value) and value > 0 for value in estimates.values())
print(pd.DataFrame({"variance": estimates, "vol_percent": {k:100*math.sqrt(v) for k,v in estimates.items()}}))

## 4. VIX futures are quoted in index points

The VIX curve is a `PriceCurve` with `kind="vol_index"`. It is a futures-term-structure input, not a variance-swap strike. VIX option terms reference a futures level and use a decimal volatility of that futures level.

In [ ]:
vix_values = {iid: price_instrument(json.dumps(inputs[iid]), market, AS_OF, model="discounting").value.amount
              for iid in ["VIX-FUTURE", "VIX-CALL"]}
print(pd.Series(vix_values, name="pv_usd"))

## Exercise — Explain the nonlinear P&L

Compare a move from 20% to 21% volatility with a move from 20% to 30%. Why does a first-order vega estimate become inaccurate for the larger move?

In [ ]:
vol_levels = np.array([0.20, 0.21, 0.30])
exact_pnl = 1_000_000*df*(vol_levels**2-0.20**2)
linear_pnl = 1_000_000*df*2*0.20*(vol_levels-0.20)
print(pd.DataFrame({"vol_percent": 100*vol_levels, "exact_usd": exact_pnl, "linear_vega_usd": linear_pnl,
              "convexity_residual_usd": exact_pnl-linear_pnl}))

### Mark a swap after 60% of its calendar life

In [ ]:
from datetime import timedelta
from finstack_quant.core.market_data import ScalarTimeSeries
seasoned_date=AS_OF+timedelta(days=219)
seasoned_market=tracks.build_market("vol",seasoned_date)
observed_dates=[AS_OF+timedelta(days=i) for i in range(220) if (AS_OF+timedelta(days=i)).weekday()<5]
closes=[(day,5200*math.exp(.012*math.sin(i*.71))) for i,day in enumerate(observed_dates)]
seasoned_market.insert_series(ScalarTimeSeries("SPX-CLOSE",closes))
seasoned=deepcopy(inputs["SPX-VARIANCE"])
# Explicitly choose a flat 20% forward-volatility assumption to isolate accrued-versus-forward accounting.
seasoned["instrument"]["spec"]["instrument_pricing_overrides"]={"market_quotes":{"implied_volatility":.20}}
seasoned_result=price_instrument(json.dumps(seasoned),seasoned_market,seasoned_date,model="discounting",
    metrics=["variance_expected","variance_realized","variance_vega","vega","variance_strike_vol"])
elapsed=(seasoned_date-AS_OF).days/365;remaining=1-elapsed
accrued=float(np.square(np.diff(np.log([value for _,value in closes]))).sum())
blended=accrued+remaining*.20**2
remaining_df=seasoned_market.get_discount("USD-OIS").df(remaining)
assert abs(elapsed-.60)<1e-12
assert abs(seasoned_result.metrics["variance_expected"]-blended)<1e-12
assert abs(seasoned_result.value.amount-1_000_000*remaining_df*(blended-.04))<.01
assert abs(seasoned_result.metrics["variance_vega"]-1_000_000*remaining_df*remaining)<.01
assert abs(seasoned_result.metrics["vega"]-seasoned_result.metrics["variance_vega"]*2*.20*.01)<.01
print(pd.Series({"elapsed_fraction":elapsed,"accrued_variance_contribution":accrued,
    "future_variance_contribution":remaining*.04,"blended_variance":blended,"pv_usd":seasoned_result.value.amount,
    **seasoned_result.metrics}))

### Compare OHLC estimators when overnight gaps matter

In [ ]:
gappy={key:[] for key in ["open","high","low","close"]};last=5200.
for i in range(40):
    opened=last*math.exp(.03*(-1)**i);closed=opened*math.exp(.008*math.cos(i))
    for key,value in [("open",opened),("high",max(opened,closed)*1.005),
                      ("low",min(opened,closed)*.995),("close",closed)]:gappy[key].append(value)
    last=closed
gap_estimates={"close_to_close":stats.realized_variance(gappy["close"])}
for method in ["parkinson","garman_klass","rogers_satchell","yang_zhang"]:
    gap_estimates[method]=stats.realized_variance_ohlc(gappy["open"],gappy["high"],gappy["low"],gappy["close"],
        method=method,annualization_factor=252.)
assert gap_estimates["close_to_close"]>gap_estimates["parkinson"]
print(pd.Series(gap_estimates,name="annualized_variance"))

### Calculate a volatility-swap convexity adjustment explicitly

In [ ]:
future_variances=np.array([.10**2,.30**2]);probabilities=np.array([.5,.5])
mean_variance=float(probabilities@future_variances)
variance_of_variance=float(probabilities@((future_variances-mean_variance)**2))
volatility_strike=float(probabilities@np.sqrt(future_variances))
variance_strike_vol=math.sqrt(mean_variance)
second_order=variance_strike_vol-variance_of_variance/(8*mean_variance**1.5)
assert volatility_strike<variance_strike_vol
print({"sqrt_expected_variance":variance_strike_vol,"expected_volatility":volatility_strike,
       "convexity_adjustment":variance_strike_vol-volatility_strike,"second_order_approximation":second_order})
print("This disclosed two-state payoff calculation is not a native volatility-swap model or a VIX-futures calibration.")

### Exercise — Translate variance and vega notionals

Convert the variance notional to dollars per volatility point at a 20% strike, then calculate terminal P&L when realized volatility is 18%. Compare with the linear approximation.

The inverse conversion must recover the starting variance notional. The exact receive-variance payoff is negative, but less negative than its local linear approximation because the payoff is convex in realized volatility.

In [ ]:
strike_vol,realized_vol=.20,.18
variance_notional=1_000_000.
vega_notional=variance_notional*2*strike_vol*.01  # USD per one volatility point near strike.
recovered=vega_notional/(2*strike_vol*.01)
terminal_pnl=variance_notional*(realized_vol**2-strike_vol**2)
linear_approx=vega_notional*((realized_vol-strike_vol)/.01)
assert abs(recovered-variance_notional)<1e-8
assert terminal_pnl<0 and terminal_pnl>linear_approx
print({"vega_notional_USD_per_vol_point":vega_notional,"terminal_exact_USD":terminal_pnl,
       "linear_USD":linear_approx,"convexity_residual_USD":terminal_pnl-linear_approx})

### Exercise — Remove the put wing

Remove every put contribution below the anchor from the same option strip. How much does the variance strike fall, and what does that say about ATM-only replication?

Keep the original anchor and correction; only remove the identified positive put contributions. The displayed difference isolates wing omission rather than changing the forward or the rest of the quadrature.

In [ ]:
put_wing=2/(T*df)*np.sum(widths[:k0_index]*np.array(option_prices[:k0_index])/strikes[:k0_index]**2)
without_puts=strip_variance-put_wing
assert put_wing>0 and without_puts<strip_variance
print({"full_variance_strike":strip_variance,"without_put_wing":without_puts,"downward_bias":put_wing})
print("Low-strike option prices receive large 1/K^2 weights. Wing truncation is model and quote dependent; matching only ATM volatility cannot validate the strip.")

### Exercise — Build the FX variance variant

Construct a EUR/USD variance swap with USD variance notional, both currency discount curves and a separate FX option surface. Confirm the result's currency and variance units.

The quote currency owns the cash payoff. The base and quote calendars jointly define observations. The forward-start date avoids pretending an unprovided fixing has already occurred. The small difference from flat 10%-volatility squared reflects the finite strip; it is displayed rather than forced to zero.

In [ ]:
from _shared.instrument_fixtures import instrument_envelope
from finstack_quant.core.market_data import VolSurface
fx_market=tracks.build_market("vol")
fx_market.insert(VolSurface("EURUSD-VAR-VOL",[.25,.5,1.,2.],[1.08*math.exp(-3+i*.05) for i in range(121)],[[.10]*121]*4))
fx_payload=instrument_envelope({"type":"fx_variance_swap","spec":{"id":"V2-EURUSD-VAR",
    "base_currency":"EUR","quote_currency":"USD","spot_id":"EURUSD-SPOT",
    "notional":{"amount":"1000000","currency":"USD"},"strike_variance":.01,
    "start_date":"2025-01-16","maturity":"2026-01-15","observation_frequency":{"count":1,"unit":"days"},
    "base_calendar_id":"weekends_only","quote_calendar_id":"weekends_only","side":"receive",
    "domestic_discount_curve_id":"USD-OIS","foreign_discount_curve_id":"EUR-OIS",
    "vol_surface_id":"EURUSD-VAR-VOL","day_count":"act_365f","attributes":{}}})
fx_market.insert_price("EURUSD-SPOT",1.08)
fx_result=price_instrument(json.dumps(fx_payload),fx_market,AS_OF,model="discounting",metrics=["variance_expected"])
assert fx_result.value.currency.code=="USD"
assert abs(fx_result.metrics["variance_expected"]-.01)<.001
print({"EURUSD_expected_variance":fx_result.metrics["variance_expected"],"USD_pv":fx_result.value.amount})

### Separate spot VIX, forward levels and roll

In [ ]:
vix_curve=market.get_vol_index_curve("VIX")
term_rows=[]
for horizon in [0.,.25,.5,1.]:
    level=vix_curve.price(horizon)
    rolled_level=vix_curve.price(max(0.,horizon-1/12))
    term_rows.append({"years":horizon,"forward_index_points":level,
                     "one_month_rolled_level_if_curve_unchanged":rolled_level,
                     "curve_roll_points":rolled_level-level})
assert term_rows[2]["forward_index_points"]>term_rows[0]["forward_index_points"]
assert term_rows[2]["curve_roll_points"]<0
print(pd.DataFrame(term_rows))
print("This fixed-curve roll is a scenario, not a forecast of the future VIX settlement. A long future's P&L also responds to changes in the entire forward curve.")
